In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier


train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

#train.head()
#train.info()

train.loc[(train['CryoSleep'] == True) , ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']] = 0
test.loc[(test['CryoSleep'] == True), ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']] = 0
train[['Deck', 'Num', 'Side']] = train['Cabin'].str.split('/', expand = True)
test[['Deck', 'Num', 'Side']] = test['Cabin'].str.split('/', expand = True)
train.drop(['Cabin', 'Name'] , axis = 1, inplace = True)
test.drop(['Cabin', 'Name'] , axis = 1, inplace = True)

#train.head()

y = train['Transported']
X = train.drop(['Transported', 'PassengerId'], axis = 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 0)

num_cols = X_train.select_dtypes(include = ['float64', 'int64']).columns
imputer = SimpleImputer(strategy = 'median')

X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
X_val[num_cols] = imputer.transform(X_val[num_cols])
test[num_cols] = imputer.transform(test[num_cols])

cat_cols = X_train.select_dtypes(include = ['object']).columns
imputer = SimpleImputer(strategy = 'most_frequent')

X_train[cat_cols] = imputer.fit_transform(X_train[cat_cols])
X_val[cat_cols] = imputer.transform(X_val[cat_cols])
test[cat_cols] = imputer.transform(test[cat_cols])

encoder = OneHotEncoder(handle_unknown = 'ignore', sparse_output = False)
encoded_train = encoder.fit_transform(X_train[cat_cols])
encoded_val = encoder.transform(X_val[cat_cols])
encoded_test = encoder.transform(test[cat_cols])
new_cols = encoder.get_feature_names_out(cat_cols)

encoded_train_df = pd.DataFrame(encoded_train, columns = new_cols, index = X_train.index)
encoded_val_df = pd.DataFrame(encoded_val, columns = new_cols, index = X_val.index)
encoded_test_df = pd.DataFrame(encoded_test, columns = new_cols, index = test.index)

X_train = X_train.drop(cat_cols, axis = 1)
X_train = pd.concat([X_train, encoded_train_df], axis = 1)

X_val = X_val.drop(cat_cols, axis = 1)
X_val = pd.concat([X_val, encoded_val_df], axis = 1)

test = test.drop(cat_cols, axis = 1)
test = pd.concat([test, encoded_test_df], axis = 1)

'''model = RandomForestClassifier(random_state = 0)
model.fit(X_train, y_train)
preds = model.predict(X_val)
score = accuracy_score(preds, y_val)
score #0.7878090856814262'''

model2 = XGBClassifier(random_state = 0, n_estimators = 1000, learning_rate = 0.05)
model2.fit(X_train, y_train)
preds2 = model2.predict(X_val)
score2 = accuracy_score(preds2, y_val)
score2

test_passenger_ids = test['PassengerId']
test_for_preds = test.drop(['PassengerId'], axis = 1)

final_preds = model2.predict(test_for_preds)

final_preds_bool = (final_preds == 1) | (final_preds == True)

output = pd.DataFrame({
    'PassengerId' : test_passenger_ids,
    'Transported' : final_preds_bool
})

output.to_csv('submission.csv', index = False)